# Feature Engineering

## Categorisation of bets based on the timing of the match
Timing of bets can be placed into 3 major categories:

1. Before the ban/pick phase
2. After the ban/pick phase and before match has started
3. Anytime before the game has ended

For this EDA, we will focus on the first two categories and the last category can be left as a future extension of the project.

In [1]:
%load_ext autoreload
%autoreload 2
from src.postgresql import get_engine
from src.pipeline.preprocessing import preprocess_df
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)



In [3]:
engine = get_engine()

In [4]:
df = pd.read_sql(
    "SELECT * FROM pro_matches",
    con=engine
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101616 entries, 0 to 101615
Data columns (total 28 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   0_hero_id        101600 non-null  float64
 1   1_hero_id        101602 non-null  float64
 2   2_hero_id        101606 non-null  float64
 3   3_hero_id        101602 non-null  float64
 4   4_hero_id        101602 non-null  float64
 5   128_hero_id      101600 non-null  float64
 6   129_hero_id      101604 non-null  float64
 7   130_hero_id      101613 non-null  float64
 8   131_hero_id      101604 non-null  float64
 9   132_hero_id      101608 non-null  float64
 10  0_account_id     101600 non-null  float64
 11  1_account_id     101602 non-null  float64
 12  2_account_id     101606 non-null  float64
 13  3_account_id     101602 non-null  float64
 14  4_account_id     101602 non-null  float64
 15  128_account_id   101600 non-null  float64
 16  129_account_id   101604 non-null  floa

# Feature Selection (Manual)

In [30]:
draft_cols = df.filter(like="_hero_id").columns
player_cols = df.filter(like="_account_id").columns
team_cols = ['radiant_name','dire_name']
label_col = 'radiant_win'
time_col = 'start_time'
uuid_col = 'match_id'

In [5]:
df = preprocess_df(df)
df.info()

Removing 2291 rows with missing values
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99325 entries, 0 to 99324
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   0_hero_id       99325 non-null  float64       
 1   1_hero_id       99325 non-null  float64       
 2   2_hero_id       99325 non-null  float64       
 3   3_hero_id       99325 non-null  float64       
 4   4_hero_id       99325 non-null  float64       
 5   128_hero_id     99325 non-null  float64       
 6   129_hero_id     99325 non-null  float64       
 7   130_hero_id     99325 non-null  float64       
 8   131_hero_id     99325 non-null  float64       
 9   132_hero_id     99325 non-null  float64       
 10  0_account_id    99325 non-null  float64       
 11  1_account_id    99325 non-null  float64       
 12  2_account_id    99325 non-null  float64       
 13  3_account_id    99325 non-null  float64       
 14  4_account_id   

# Feature Engineering

## Create Team Level Features

In [4]:
from collections import deque

In [5]:
def calculate_win_rate(team_histories: list, team_name: str) -> float:
    if not team_histories:
        return 0.5
    
    win = 0
    for match in team_histories:
        if match['radiant_name'] == team_name and match['radiant_win']:
            win += 1
        elif match['dire_name'] == team_name and not match['radiant_win']:
            win += 1
        
    return win/ len(team_histories)
        

In [6]:


def update_team_history(team_histories, radiant, dire, match):
    if radiant not in team_histories:
        team_histories[radiant] = deque(maxlen=10)
    if dire not in team_histories:
        team_histories[dire] = deque(maxlen=10)
        
    team_histories[radiant].append(match)
    team_histories[dire].append(match)
    
def update_matchup_history(matchup_histories, radiant, dire, match):
    if (radiant, dire) not in matchup_histories:
        matchup_histories[(radiant, dire)] = deque(maxlen=10)
    
    matchup_histories[(radiant,dire)].append(match)

In [7]:
def calculate_matchup(matchup_histories: list, team_name:str) -> float:
    if not matchup_histories:
        return 0.5
    
    win = 0
    for match in matchup_histories:
        if match['radiant_name'] == team_name and match['radiant_win']:
            win += 1
        elif match['dire_name'] == team_name and not match['radiant_win']:
            win += 1
    
    return win / len(matchup_histories)

In [18]:
team_histories = {}
matchup_histories = {}
team_level_features = []

for _, match in df.iterrows():
    radiant_team = match['radiant_name']
    dire_team = match['dire_name']
    
    # Calculate features for each row
    radiant_win_rate = calculate_win_rate(team_histories.get(radiant_team, []), radiant_team)
    dire_win_rate = calculate_win_rate(team_histories.get(dire_team, []), dire_team)
    
    all_matches = list(matchup_histories.get((radiant_team, dire_team), [])) \
                    + list(matchup_histories.get((dire_team, radiant_team), []))
    matchup_rate = calculate_matchup(all_matches, radiant_team)
    
    # append features to results
    team_level_features.append({
        'match_id': match['match_id'],
        'radiant_win_rate': radiant_win_rate,
        'dire_win_rate': dire_win_rate,
        'radiant_dire_matchup': matchup_rate
    })
    
    # update history dictionaries
    update_team_history(team_histories, radiant_team, dire_team, match)
    update_matchup_history(matchup_histories, radiant_team, dire_team, match)
    
    
team_level_features
    
    

[{'match_id': 5999176266,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999201501,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999214195,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999249937,
  'radiant_win_rate': 1.0,
  'dire_win_rate': 0.0,
  'radiant_dire_matchup': 1.0},
 {'match_id': 5999283181,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999407636,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999433909,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999441172,
  'radiant_win_rate': 0.0,
  'dire_win_rate': 1.0,
  'radiant_dire_matchup': 0.0},
 {'match_id': 5999477345,
  'radiant_win_rate': 1.0,
  'dire_win_rate': 0.0,
  'radiant_dire_matchup': 1.0},
 {'match_id': 59994

In [1]:
from sqlmodel import Session
from database.schemas.features import TeamFeatures

In [20]:
model_fields = {
    name for name, field in TeamFeatures.__fields__.items()
    if not name.startswith('_')
}

model_fields

{'dire_win_rate', 'match_id', 'radiant_dire_matchup', 'radiant_win_rate'}

In [33]:
# Store to database:

with Session(engine) as session:
    for row in team_level_features:
        # 'row' is a pandas Series, which works similarly to a dictionary
        filtered_data = {
            field: row[field]
            for field in model_fields
            if field in row
        }
        
        # Create the model instance with the filtered data
        team_features = TeamFeatures(**filtered_data)
        session.merge(team_features)
    
    session.commit()

## Create hero level feature 

In [6]:
import yaml
CONSTANTS_FILE_PATH = '../constants/constants.yml'
try:
    with open(CONSTANTS_FILE_PATH, 'r') as file:
        data = yaml.safe_load(file) or {}
        hero_dict = data.get('HEROES_CONSTANTS', {})
        if not hero_dict or not isinstance(hero_dict, dict):
            raise ValueError("Unable to load hero constants or they are not in a valid format.")
except FileNotFoundError:
    print(f"'{CONSTANTS_FILE_PATH}' does not exist!")
    
hero_dict

{1: 'Anti-Mage',
 2: 'Axe',
 3: 'Bane',
 4: 'Bloodseeker',
 5: 'Crystal Maiden',
 6: 'Drow Ranger',
 7: 'Earthshaker',
 8: 'Juggernaut',
 9: 'Mirana',
 10: 'Morphling',
 11: 'Shadow Fiend',
 12: 'Phantom Lancer',
 13: 'Puck',
 14: 'Pudge',
 15: 'Razor',
 16: 'Sand King',
 17: 'Storm Spirit',
 18: 'Sven',
 19: 'Tiny',
 20: 'Vengeful Spirit',
 21: 'Windranger',
 22: 'Zeus',
 23: 'Kunkka',
 25: 'Lina',
 26: 'Lion',
 27: 'Shadow Shaman',
 28: 'Slardar',
 29: 'Tidehunter',
 30: 'Witch Doctor',
 31: 'Lich',
 32: 'Riki',
 33: 'Enigma',
 34: 'Tinker',
 35: 'Sniper',
 36: 'Necrophos',
 37: 'Warlock',
 38: 'Beastmaster',
 39: 'Queen of Pain',
 40: 'Venomancer',
 41: 'Faceless Void',
 42: 'Wraith King',
 43: 'Death Prophet',
 44: 'Phantom Assassin',
 45: 'Pugna',
 46: 'Templar Assassin',
 47: 'Viper',
 48: 'Luna',
 49: 'Dragon Knight',
 50: 'Dazzle',
 51: 'Clockwerk',
 52: 'Leshrac',
 53: "Nature's Prophet",
 54: 'Lifestealer',
 55: 'Dark Seer',
 56: 'Clinkz',
 57: 'Omniknight',
 58: 'Enchantress

In [7]:
DRAFT_COLS = [
    '0_hero_id', '1_hero_id', '2_hero_id', '3_hero_id', '4_hero_id',
    '128_hero_id', '129_hero_id', '130_hero_id', '131_hero_id', '132_hero_id'
]

In [8]:
# 1. Add error handling for hero mapping
df[DRAFT_COLS] = df[DRAFT_COLS].map(lambda x: hero_dict.get(x, "unknown_hero"))
df[DRAFT_COLS]

,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id
0,Wraith King,Witch Doctor,Earth Spirit,Leshrac,Dark Seer,Batrider,Lifestealer,Earthshaker,Timbersaw,Grimstroke
1,Ancient Apparition,Enchantress,Magnus,Ember Spirit,Tiny,Oracle,Storm Spirit,Broodmother,Nyx Assassin,Drow Ranger
2,Centaur Warrunner,Hoodwink,Razor,Grimstroke,Gyrocopter,Kunkka,Axe,Medusa,Snapfire,Witch Doctor
3,Void Spirit,Brewmaster,Terrorblade,Snapfire,Abaddon,Rubick,Centaur Warrunner,Ember Spirit,Medusa,Ancient Apparition
4,Enchantress,Timbersaw,Tiny,Wraith King,Oracle,Faceless Void,Ancient Apparition,Centaur Warrunner,Snapfire,Templar Assassin
...,...,...,...,...,...,...,...,...,...,...
99320,Jakiro,Tidehunter,Lina,Rubick,Tiny,Anti-Mage,Pudge,Sand King,Shadow Fiend,Crystal Maiden
99321,Zeus,Pudge,Wraith King,Sniper,Death Prophet,Silencer,Queen of Pain,Night Stalker,Tiny,Weaver
99322,Tinker,Pangolier,Tiny,Leshrac,Lifestealer,Dragon Knight,Ember Spirit,Ancient Apparition,Faceless Void,Dark Willow
99323,Ancient Apparition,Magnus,Pudge,Puck,Sven,Abaddon,Wraith King,Invoker,Shadow Fiend,Timbersaw


In [11]:
selected_cols = DRAFT_COLS + ['match_id']
heroes_features = df[selected_cols]

In [12]:
from sqlmodel import Session
from database.schemas.features import HeroFeatures

In [15]:
with Session(engine) as session:
    for _, row in heroes_features.iterrows():
        # Filter the row data to only include fields in the model
        # Convert to dict first to make it easier to filter
        row_dict = dict(row)
        match_id = row_dict['match_id']
        hero_picks = []
        
        for column, value in row_dict.items():
            if column in DRAFT_COLS:
                hero_picks.append(value)
                
        hero_features = HeroFeatures(
            match_id=match_id,
            hero_picks=hero_picks
        )
        
        session.merge(hero_features)
    
    session.commit()

#### Feature Transformation

In [24]:
from sklearn.preprocessing import MultiLabelBinarizer

In [25]:
hero_features = pd.read_sql(
    "SELECT * FROM hero_features",
    con=engine
)

hero_features

,match_id,hero_picks
0,6002092367,"[Kunkka, Sven, Ancient Apparition, Mars, Shado..."
1,6003543228,"[Spectre, Void Spirit, Mars, Enchantress, Elde..."
2,6061828036,"[Hoodwink, Disruptor, Magnus, Puck, Spectre, L..."
3,6067799696,"[Spectre, Magnus, Nyx Assassin, Mirana, Dragon..."
4,6074484431,"[Dark Willow, Drow Ranger, Kunkka, Broodmother..."
...,...,...
99320,8230206445,"[Slark, Leshrac, Mars, Muerta, Silencer, Abadd..."
99321,8230245010,"[Vengeful Spirit, Tinker, Kunkka, Phoenix, Mor..."
99322,8230270883,"[Shadow Shaman, Shadow Fiend, Leshrac, Abaddon..."
99323,8230283967,"[Gyrocopter, Invoker, Beastmaster, Tusk, Silen..."


In [26]:
ALL_HEROES = list(hero_dict.values())

In [ ]:
mlb = MultiLabelBinarizer(classes=ALL_HEROES)
hero_matrix = mlb.fit_transform(hero_features['hero_picks'])
features = pd.DataFrame(hero_matrix, columns=mlb.classes_)
features.insert(0, 'match_id', hero_features['match_id'].values)


In [28]:
features

,match_id,Anti-Mage,Axe,Bane,Bloodseeker,Crystal Maiden,Drow Ranger,Earthshaker,Juggernaut,Mirana,Morphling,Shadow Fiend,Phantom Lancer,Puck,Pudge,Razor,Sand King,Storm Spirit,Sven,Tiny,Vengeful Spirit,Windranger,Zeus,Kunkka,Lina,Lion,Shadow Shaman,Slardar,Tidehunter,Witch Doctor,Lich,Riki,Enigma,Tinker,Sniper,Necrophos,Warlock,Beastmaster,Queen of Pain,Venomancer,Faceless Void,Wraith King,Death Prophet,Phantom Assassin,Pugna,Templar Assassin,Viper,Luna,Dragon Knight,Dazzle,...,Brewmaster,Shadow Demon,Lone Druid,Chaos Knight,Meepo,Treant Protector,Ogre Magi,Undying,Rubick,Disruptor,Nyx Assassin,Naga Siren,Keeper of the Light,Io,Visage,Slark,Medusa,Troll Warlord,Centaur Warrunner,Magnus,Timbersaw,Bristleback,Tusk,Skywrath Mage,Abaddon,Elder Titan,Legion Commander,Techies,Ember Spirit,Earth Spirit,Underlord,Terrorblade,Phoenix,Oracle,Winter Wyvern,Arc Warden,Monkey King,Dark Willow,Pangolier,Grimstroke,Hoodwink,Void Spirit,Snapfire,Mars,Ring Master,Dawnbreaker,Marci,Primal Beast,Muerta,Kez
0,6002092367,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0
1,6003543228,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
2,6061828036,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,6067799696,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,6074484431,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99320,8230206445,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0
99321,8230245010,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
99322,8230270883,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0
99323,8230283967,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0


## Feature Crossing between players and heros



In [35]:
df_melted_players = pd.melt(df.copy(), id_vars=['start_time','radiant_win','match_id'],
                            value_vars=player_cols, 
                            var_name='player_position',
                            value_name='account_id')

df_melted_players

,start_time,radiant_win,match_id,player_position,account_id
0,2021-05-17 21:06:51,True,5999176266,0_account_id,452400903.0
1,2021-05-17 21:40:35,True,5999201501,0_account_id,86818655.0
2,2021-05-17 22:02:01,False,5999214195,0_account_id,118207269.0
3,2021-05-17 23:02:25,False,5999249937,0_account_id,874542740.0
4,2021-05-18 00:02:06,False,5999283181,0_account_id,126469785.0
...,...,...,...,...,...
993245,2025-03-27 02:05:35,True,8230656847,132_account_id,120488420.0
993246,2025-03-27 02:44:14,True,8230677659,132_account_id,451153940.0
993247,2025-03-27 03:11:15,True,8230693148,132_account_id,392385140.0
993248,2025-03-27 03:28:22,False,8230701740,132_account_id,244664524.0


In [33]:
df_melted_heroes = pd.melt(df.copy(), id_vars=['start_time','radiant_win','match_id'],
                            value_vars=draft_cols,
                            var_name='hero_position',
                            value_name='hero_name')

df_melted_heroes

,start_time,radiant_win,match_id,hero_position,hero_name
0,2021-05-17 21:06:51,True,5999176266,0_hero_id,Wraith King
1,2021-05-17 21:40:35,True,5999201501,0_hero_id,Ancient Apparition
2,2021-05-17 22:02:01,False,5999214195,0_hero_id,Centaur Warrunner
3,2021-05-17 23:02:25,False,5999249937,0_hero_id,Void Spirit
4,2021-05-18 00:02:06,False,5999283181,0_hero_id,Enchantress
...,...,...,...,...,...
993245,2025-03-27 02:05:35,True,8230656847,132_hero_id,Crystal Maiden
993246,2025-03-27 02:44:14,True,8230677659,132_hero_id,Weaver
993247,2025-03-27 03:11:15,True,8230693148,132_hero_id,Dark Willow
993248,2025-03-27 03:28:22,False,8230701740,132_hero_id,Timbersaw


In [36]:
df_melted_players['player_num'] = df_melted_players['player_position'].apply(lambda x: x.split('_')[0])
df_melted_heroes['hero_num'] = df_melted_heroes['hero_position'].apply(lambda x: x.split('_')[0])

In [ ]:
df_combined = pd.merge(df_melted_players, df_melted_heroes, 
                       left_on=['start_time', 'radiant_win','match_id', 'player_num'], 
                       right_on=['start_time', 'radiant_win','match_id', 'hero_num'])

df_combined = df_combined.sort_values(by='start_time', ascending=False)
df_combined


In [ ]:
# 2. Determine if a player won based on their position and match outcome
df_combined['player_won'] = ((df_combined['player_num'].astype(int) < 5) & df_combined['radiant_win']) | \
                           ((df_combined['player_num'].astype(int) >= 5) & ~df_combined['radiant_win'])


In [ ]:
# 2. Create a key for each account_id and hero combination
# Use hero_name (not hero_num) to identify unique heroes
df_combined['account_hero_key'] = df_combined['account_id'].astype(str) + '_' + df_combined['hero_name'].astype(str)

In [ ]:
# 3. Sort data chronologically
df_sorted = df_combined.sort_values(by=time_col)

In [ ]:
win_rates = {}
for key, group in df_sorted.groupby('account_hero_key'):
    for i, row in group.iterrows():
        match_id = row['match_id']
        player_num = row['player_num']
        current_time = row[time_col]
        
        # Find previous matches for this player-hero combo
        previous_matches = group[group[time_col] < current_time]
        
        # Calculate win rate from previous matches
        if len(previous_matches) > 0:
            previous_10 = previous_matches.sort_values(by=time_col, ascending=False).head(10)
            win_rate = previous_10['player_won'].mean()
        else:
            win_rate = 0.5
            
        win_rates[(match_id, player_num)] = win_rate

In [ ]:
# 5. Apply calculated win rates to dataframe
df_combined['win_rate'] = df_combined.apply(
    lambda row: win_rates.get((row['match_id'], row['player_num']), 0.5),
    axis=1
)

In [ ]:
# 6. Create column names based on player and hero positions
df_combined['player_hero_win_rate_col'] = (
    'player_hero_' + df_combined['player_num'].astype(str) + '_win_rate'
)

In [ ]:
# 7. Create final pivot table with position-based columns
player_hero_features = df_combined.pivot(
    index='match_id', 
    columns='player_hero_win_rate_col', 
    values='win_rate'
).reset_index()

player_hero_features

In [ ]:
player_hero_features

In [ ]:
from database.schemas.features import PlayerHeroFeature

In [ ]:
model_fields = {
    name for name in PlayerHeroFeature.model_fields.keys()
    if not name.startswith('_')
}

model_fields

In [ ]:
def store_player_hero_features(engine, player_hero_feature: pd.DataFrame):
    """
    Store the calculated player-hero win rates to the database
    
    Parameters:
    - engine: SQLAlchemy engine
    - player_hero_feature: DataFrame with match_id and win rate columns
    """
    # Convert DataFrame to list of dictionaries (one dict per match)
    records = player_hero_feature.to_dict(orient="records")
    
    # Create PlayerHeroFeature objects and insert them
    with Session(engine) as session:
        # For each match record
        for record in records:
            # Create a new PlayerHeroFeature instance
            player_hero_feature_obj = PlayerHeroFeature(
                match_id=record["match_id"],
                player_hero_0_win_rate=record["player_hero_0_win_rate"],
                player_hero_1_win_rate=record["player_hero_1_win_rate"],
                player_hero_2_win_rate=record["player_hero_2_win_rate"], 
                player_hero_3_win_rate=record["player_hero_3_win_rate"],
                player_hero_4_win_rate=record["player_hero_4_win_rate"],
                player_hero_128_win_rate=record["player_hero_128_win_rate"],
                player_hero_129_win_rate=record["player_hero_129_win_rate"],
                player_hero_130_win_rate=record["player_hero_130_win_rate"],
                player_hero_131_win_rate=record["player_hero_131_win_rate"],
                player_hero_132_win_rate=record["player_hero_132_win_rate"]
            )
            
            # Use merge instead of add
            session.merge(player_hero_feature_obj)
        
        # Commit all records at once
        try:
            session.commit()
            print(f"Successfully stored {len(records)} player-hero feature records")
        except Exception as e:
            session.rollback()
            print(f"Error storing player-hero features: {str(e)}")
            
store_player_hero_features(engine, player_hero_features)